# Imports

In [1]:
from langchain.text_splitter import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

In [2]:
import os
import pandas as pd
import config
import json

In [3]:
os.environ["OPENAI_API_KEY"] = config.OPENAI_API_KEY
os.environ["OPENAI_API_BASE"] = config.OPENAI_API_BASE

# Create ChromaDB

In [4]:
embedding_function = OpenAIEmbeddings()
# Initialize ChromaDB in LangChain
vector_db = Chroma(
    collection_name="housing_data",
    embedding_function=embedding_function,
    persist_directory="chroma_storage"  # Ensures persistence
)

In [5]:
df = pd.read_csv('dummy_housing_data_v2.csv', index_col=0)
df

,id,neighborhood_name,price,bedrooms,bathrooms,house_size,year_built,description,neighborhood_description
0,2,Willow Creek,650000,4,3.5,3000,2005,Discover the perfect family home in the desira...,Willow Creek is known for its top-rated school...
1,3,Riverfront Estates,950000,5,4.0,5000,2010,Luxury living awaits in the prestigious Riverf...,"Riverfront Estates offers waterfront views, pr..."
2,4,Sunny Hills,400000,3,2.5,1800,1998,Welcome home to the charming neighborhood of S...,Sunny Hills is a family-friendly community wit...
3,5,Oak Ridge,750000,4,3.0,2800,2008,Live the ultimate suburban lifestyle in the so...,"Oak Ridge boasts tree-lined streets, community..."
4,6,Maple Grove,550000,3,2.5,2400,2000,Step into this meticulously maintained home in...,"Maple Grove offers tree-lined streets, communi..."
5,7,Pinecrest Heights,300000,2,1.5,1500,1980,"Cozy up in this charming 2-bedroom, 1.5-bathro...",Pinecrest Heights is a hidden gem that offers ...
6,8,Meadowbrook Ridge,850000,4,3.0,3200,2015,Indulge in luxury living in the upscale Meadow...,Meadowbrook Ridge is an exclusive community kn...
7,9,Cedar Grove,500000,3,2.0,2200,2003,Welcome to the tranquil neighborhood of Cedar ...,Cedar Grove is a close-knit community with fri...
8,10,Sunset Hills,700000,4,3.5,2600,2007,Experience luxury living in the prestigious Su...,"Sunset Hills is known for its upscale homes, p..."
9,11,Riverside Park,450000,3,2.5,1900,1995,Escape to the peaceful retreat of Riverside Pa...,Riverside Park is a nature lover's paradise wi...


In [6]:
# Assume `df` is your Pandas DataFrame with listings
def dataframe_to_documents(df):
    documents = []
    for _, row in df.iterrows():
        text = f"{row['description']}\n"\
               f"House size: {row['bedrooms']} bedrooms, {row['bathrooms']} bathrooms, {row['house_size']} sqft.\n"\
               f"Built in {row['year_built']}.\n"\
               f"Located in {row['neighborhood_name']}. {row['neighborhood_description']}"
        
        metadata = row.drop(['description','neighborhood_description','id']).to_dict()  # Store metadata for filtering
        documents.append(Document(page_content=text, metadata=metadata))
    return documents

docs = dataframe_to_documents(df)

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

split_docs = splitter.split_documents(docs)

In [8]:
created_ids = vector_db.add_documents(docs)
print(len(created_ids))

10


# Find Best Fit Housing Listings

## Load Test Q&A 

In [9]:
with open('qa_data.json', 'r') as file:
     qa_data = json.load(file)
print(qa_data)

{'questions': ['How many bed- and bathrooms do you want?', 'How many square feet?', 'How old can the house be?', 'What are the 3 most important things in the house?', 'Are there any amenities you want?', 'What kind of neighborhood do you want to live in?', 'How urban do you want your neighborhood to be?'], 'customer1': ['I want a three-bedroom, two-bathroom house.', 'Between 2000 to 3000 sqft.', 'I think it should be built in 1985 or later.', 'A spacious kitchen, a cozy living room, big windows.', 'A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.', 'A quiet neighborhood with good local schools, and convenient shopping options and easy access to major highways.', 'A balance between suburban tranquility and access to urban amenities like restaurants and theaters.'], 'customer2': ['A luxurious 5-bedroom, 3-bathroom house.', '5000 sqft or more.', 'Relatively new, built 2000 or later.', 'A gourmet kitchen, a winter garden, high ceilings.', 'A big ga

In [10]:
def create_qa(questions, answers):
    qa = ""
    for i in range(len(questions)):
        qa += "Q: " + questions[i] + "\nA: " + answers[i] + "\n"

    return qa

In [11]:
query_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""
            You are a real estate agent trying to sell a house to a customer.
            Based on the housing listings in the context, extract 3 of the listings that fit the customer requirements the most.\n
            <context>
            {context}
            </context>
            The customer requests have to be identified from the below user chat history: \n
            {input}

            Rewrite the information of each housing listing to fit the preferences of the customer. 
            Do not add anything that is not given information.
        
            Make it sound appealing, engaging, and personalized. 
            Directly speak to the customer and point out in a structured manner how the house aligns with the customers mentioned preferences.

            Then, add another section that compares the three houses which each other in a structured way to make them more comparable.
            """
)

In [12]:
# Define the LLM model with a retriever vector-db

model_name = "gpt-3.5-turbo"
temperature = 0.3
llm = ChatOpenAI(model_name=model_name, temperature=temperature)
retriever = vector_db.as_retriever()
document_chain = create_stuff_documents_chain(llm, query_prompt)

retrieval_chain = create_retrieval_chain(retriever, document_chain)

## Results

In [16]:
for number in range(1,4):
    print(f"***************************************Customer {number}*************************************")
    user_input = create_qa(qa_data['questions'], qa_data['customer%s'%number])
    print(user_input)
    print("*******************************************************")
    response = retrieval_chain.invoke({"input": user_input})
    print(response["answer"])

***************************************Customer 1*************************************
Q: How many bed- and bathrooms do you want?
A: I want a three-bedroom, two-bathroom house.
Q: How many square feet?
A: Between 2000 to 3000 sqft.
Q: How old can the house be?
A: I think it should be built in 1985 or later.
Q: What are the 3 most important things in the house?
A: A spacious kitchen, a cozy living room, big windows.
Q: Are there any amenities you want?
A: A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.
Q: What kind of neighborhood do you want to live in?
A: A quiet neighborhood with good local schools, and convenient shopping options and easy access to major highways.
Q: How urban do you want your neighborhood to be?
A: A balance between suburban tranquility and access to urban amenities like restaurants and theaters.

*******************************************************
Listing 1:
Welcome to your dream home in the charming neighborhood of 